In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle
import rpy2.situation
for row in rpy2.situation.iter_info():
    print(row)
from rpy2.robjects import r
from rpy2.robjects.packages import importr
%matplotlib qt
from local_functions import pix_to_deg,read_data,calculate_bf,plot_bfs,shrink_axes,make_subplots, inset_axes
from statsmodels.formula.api import probit
from statsmodels.formula.api import ols
from scipy.stats import linregress
from scipy.interpolate import interp1d


import rpy2.robjects as ro
ro.r('.libPaths(c(.libPaths(), "/Library/Frameworks/R.framework/Versions/4.6/Resources/library"))')
bf_package=importr('BayesFactor')

# A4 dimensions in inches
a4_width = 8.27 
a4_height = 11.69 

task = 'exp2'

subj_colors = sns.color_palette("Spectral",21)
colors = sns.diverging_palette(250, 30, l=70, n=9,center='dark')



CFFI extension type
  ABI: PRESENT
  API: PRESENT
  Environment variable: RPY2_CFFI_MODE=ANY
  Loaded: CFFI_MODE.API
rpy2 version:
rpy2.rinterface: 3.6.6
Python version:
3.14.4 | packaged by Anaconda, Inc. | (main, Apr 14 2026, 17:01:42) [Clang 20.1.8 ]
Looking for R's HOME:
  Environment variable R_HOME: None
  Calling `R RHOME`: /Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/R
  Environment variable R_LIBS_USER: None
R's value for LD_LIBRARY_PATH:
  
R version:
  In the PATH: R version 4.5.3 (2026-03-11) -- "Reassured Reassurer"
  Loading R library from rpy2: OK
Additional directories to load R packages from:
  None
C extension compilation:
  include:
    ['/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/R/include']
  libraries:
    ['R', 'bz2', 'z', 'dl', 'm', 'iconv', 'icuuc', 'icui18n']
  library_dirs:
    ['/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib', '/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/R/lib', '/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib']
  extra_compile_

### read data

In [2]:
all_epochs = []
bids_dir = '/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data'
subjids = [i for i in np.arange(1,21) if i!=9]
all_epochs = read_data(bids_dir,subjids,task = 'exp2')

Reading /Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp2/derivatives/preprocessed/sub-S01_exp2_eyetracking-preproc-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -500.00 ...    2500.00 ms
        0 CTF compensation matrices available
Adding metadata with 25 columns
216 matching events found
No baseline correction applied
0 projection items activated
Reading /Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp2/derivatives/preprocessed/sub-S02_exp2_eyetracking-preproc-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -500.00 ...    2500.00 ms
        0 CTF compensation matrices available
Adding metadata with 25 columns
216 matching events found
No baseline correction applied
0 projection items activated
Reading /Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp2/derivatives/preprocessed/sub-

### make plot

In [3]:
plt.close('all')
fig,axs = plt.subplots(2,2,num=1,figsize=(a4_width, a4_height*0.7))


epos = all_epochs[0].copy()
x_pos = epos.metadata.occlusion_start_time.unique()[0]

# Labels for the subplots
labels = ['(a)', '(b)', '(c)','(d)']
for i, (ax, label) in enumerate(zip(axs.flat, labels)):
    pos = ax.get_position()  
    x = pos.x0 - 0.05
    y = pos.y1 + 0.02 
    fig.text(x, y, label, fontsize=12, fontweight='bold', va='top', ha='right')

#************************************
### (A) TRIAL
#************************************
ax = shrink_axes(axs[0][0],0.95,0.8)

## load some sample data
x_ind = np.where([i=='xpos' for i in all_epochs[0].info.ch_names])[0][0]
subj = 4 
sample_p = 0
epos = all_epochs[sample_p].copy()
x_idx = np.where([i=='xpos' for i in epos.ch_names])[0][0]
epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1


## plot the trajectories for all conditions 
traj = np.linspace(np.min(epos.metadata.end_pos_deg.unique()),np.max(epos.metadata.end_pos_deg.unique()),epos.metadata['n_frames'].unique()[0])
total_traj_time = epos.metadata['n_frames'].unique()[0]*1/epos.metadata['monitor_refreshrate'].unique()[0]
traj_tv = np.arange(0,total_traj_time,1/epos.metadata['monitor_refreshrate'].unique()[0])


traj_occluder = traj.copy()
traj_occluder[(traj >= np.min(epos.metadata.occluder_start_pos_deg.unique())) & (traj < np.max(epos.metadata.occluder_start_pos_deg.unique()))]=np.nan

pre_occ_traj = traj[traj<=np.min(epos.metadata.occluder_start_pos_deg.unique())]
post_occ_traj = traj[traj>np.max(epos.metadata.occluder_start_pos_deg.unique())]
standard_occ_length = len(np.where(np.isnan(traj_occluder))[0])
n_occ_frames = np.flip(np.sort(epos.metadata.jump_size_frames.unique()))

for i in range(len(n_occ_frames)):
    curr_traj = list(pre_occ_traj)+list(np.ones(int(standard_occ_length+n_occ_frames[i]))*np.nan)+list(post_occ_traj)

    tmp_tv = np.linspace(0,1/60*len(curr_traj),len(curr_traj))
    if i == len(n_occ_frames)//2:
        ax.plot(tmp_tv,curr_traj,color=colors[i],alpha=1,lw=3,zorder = 100)
    else:
        ax.plot(tmp_tv,curr_traj,color=colors[i],alpha=1,lw=3)
    
    
    occ_start_idx = np.where(np.isnan(curr_traj))[0][0]
    occ_end_idx = np.where(np.isnan(curr_traj))[0][-1]

    x = [tmp_tv[occ_start_idx-1],tmp_tv[occ_end_idx+1]]
    y = [curr_traj[occ_start_idx-1],curr_traj[occ_end_idx+1]]

    for ii in range(0, len(x), 2):
        ax.plot(x[ii:ii+2], y[ii:ii+2], color=colors[i],linestyle=':')

ax.set_xlabel('time since trial start (s)')
ax.set_ylabel('x-position object (dva)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlim((np.float64(-0.65), np.float64(3)))

ax.vlines(x_pos,-np.max(epos.metadata.occluder_start_pos_deg),np.max(epos.metadata.occluder_start_pos_deg),'k',linestyles='-',lw=2)
ax.hlines(-np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.hlines(np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.annotate('occluder', xy=(x_pos, -5),xytext=(x_pos, -8),ha='center', va='top', fontsize=9,arrowprops=dict(arrowstyle='->', color='k'),clip_on=False)

#************************************
### (B) BEHAVIOUR
#************************************
ax = shrink_axes(axs[0][1],0.85,0.8)
## plot the behavioural performance, model fits, and PSEs
all_pse, all_pred_probs = [],[]

for subj in range(19):
    dat = all_epochs[subj].metadata.copy()


    dat['response_numeric'] = dat['response'].map({'too early': 1, 'too late': 0})

    # ignoring different motion directions [L->R and R->L]
    dat.loc[dat.direction==2,'start_pos_deg'] = dat.loc[dat.direction==2,'start_pos_deg']*-1
    dat.loc[dat.direction==2,'start_pos'] = dat.loc[dat.direction==2,'start_pos']*-1
    dat.loc[dat.direction==2,'jump_size_pix'] = dat.loc[dat.direction==2,'jump_size_pix']*-1
    dat.loc[dat.direction==2,'jump_size_deg'] = dat.loc[dat.direction==2,'jump_size_deg']*-1

    

    model = probit("response_numeric ~ jump_size_deg",dat).fit()
    n = 1001
    x = pd.DataFrame({'jump_size_deg': np.linspace(dat["jump_size_deg"].min(), dat["jump_size_deg"].max(),n)})
    pred_probability = model.predict(x)
    pse = np.argmin(np.abs(pred_probability-0.5))-n/2
    tmp=x.to_numpy()
    pse = tmp[np.argmin(np.abs(pred_probability-0.5))]


    response_grouped = dat.groupby('jump_size_deg').mean(numeric_only=True)['response_numeric']
    
    [ax.plot(response_grouped.index.to_numpy()[i],response_grouped.to_numpy()[i]*100,marker='.',markersize=8,alpha=1,linestyle="None",color=colors[i],markeredgecolor=None) for i in range(len(response_grouped))]
    

    # group_data
    all_pse.append(pse)
    all_pred_probs.append(pred_probability)

[ax.plot(x,pred_probability*100,color='k',alpha=0.2) for pred_probability in all_pred_probs]


ax.hlines(50,xmin=np.min(x),xmax=np.max(x),colors='r',linestyles=':',zorder=0)
ax.vlines(0,ymin=0,ymax=100,colors='r',linestyles=':',zorder=0)
ax.plot(x,np.mean(all_pred_probs,axis=0)*100,color='k',alpha=1,lw=3)

n_jumpsizes = len(dat.jump_size_frames.unique())
max_reappearance_delay = np.sort(dat.jump_size_frames.unique()*1/dat.monitor_refreshrate.unique())[-1]
ax.set_xticks([x.to_numpy()[0][0],0,x.to_numpy()[-1][0]])
ax.set_xticklabels([f'delayed\nreappearance ({np.round(max_reappearance_delay,2)} s)','on time',f'early\nreappearance ({np.round(max_reappearance_delay,2)} s)'],fontsize=9)
ax.set_ylabel('% "too early" responses')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


#### axis inset showing the PSE distribution
pos = ax.get_position()

ax_inset = plt.gcf().add_axes([pos.x0, pos.y0 + pos.height * 0.95, pos.width, pos.height * 0.2])

ax_inset.boxplot([np.array(all_pse).flatten()], vert=False, widths=0.5,patch_artist=True,boxprops=dict(facecolor='white', color='k'),medianprops=dict(color='k'),whiskerprops=dict(color='k'),capprops=dict(color='k'),flierprops=dict(marker='x', color='k', markersize=4))

ax_inset.set_xlim(ax.get_xlim())
ax_inset.axis('off')

right_end = max([l.get_xdata().max() for l in ax_inset.lines])

### Bayes Factors
bf_pse = calculate_bf(np.array(all_pse))
bf_print = str(np.round(bf_pse,3)[0])
# ax_inset.annotate(f'Distribution: PSEs\nBF={bf_print}', xy=(right_end, 1),xytext=(right_end + 1, 1),  va='center', ha='left', fontsize=9, arrowprops=dict(arrowstyle='->', color='k'))
ax_inset.annotate(f'Distribution: PSEs\nBF={bf_print}', xy=(right_end, 1),xytext=(right_end + 1, 1),  va='center', ha='left', fontsize=9, arrowprops=dict(arrowstyle='->', color='k'))


print(f'mean pse  = {np.mean(all_pse)}, BF = 10^{np.log10(np.array(bf_pse))}')
print(bf_pse)


#************************************
### (C) EYETRACKING -- individual
#************************************

ax = shrink_axes(axs[1][0],0.95,0.8)

select_p = 1
toplot_group = []
for subj in range(19):
    epos = all_epochs[subj].copy()

    x_idx = np.where([i=='xpos' for i in epos.ch_names])[0][0]
    epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

    # ignoring different motion directions [L->R and R->L]
    epos.metadata.loc[epos.metadata.direction==2,'start_pos_deg'] = epos.metadata.loc[epos.metadata.direction==2,'start_pos_deg']*-1
    epos.metadata.loc[epos.metadata.direction==2,'start_pos'] = epos.metadata.loc[epos.metadata.direction==2,'start_pos']*-1
    epos.metadata.loc[epos.metadata.direction==2,'jump_size_pix'] = epos.metadata.loc[epos.metadata.direction==2,'jump_size_pix']*-1
    epos.metadata.loc[epos.metadata.direction==2,'jump_size_deg'] = epos.metadata.loc[epos.metadata.direction==2,'jump_size_deg']*-1


    toplot = epos._data[:,x_ind,:]

    jum_sizes = np.sort(epos.metadata.jump_size_deg.unique())
    occlusion_end_times = np.flip(np.sort(epos.metadata.loc[:,'occlusion_end_time'].unique()))
    toplot_group_alljumps = []
    for js_i in np.arange(9):
        js = jum_sizes[js_i]
        toplot_js = toplot[epos.metadata.jump_size_deg==js,:]
        toplot_group_alljumps.append(np.nanmean(toplot_js,axis=0))


        if subj == select_p:
            [ax.plot(epos.times,toplot_js[i,:],alpha=0.2,lw=0.5,color=colors[js_i],zorder=0) for i in range(toplot_js.shape[0])]
            ax.plot(epos.times,np.nanmean(toplot_js,axis=0),alpha=1,lw=2,color=colors[js_i],zorder=1)
            ax.plot(occlusion_end_times[js_i],-12,'^',color=colors[js_i],markersize=10,zorder=10,clip_on=1) 

    toplot_group.append(toplot_group_alljumps)

    if subj == select_p:
        ax.vlines(x_pos,-np.max(epos.metadata.occluder_start_pos_deg),np.max(epos.metadata.occluder_start_pos_deg),'k',linestyles='-',lw=2)
        ax.hlines(-np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
        ax.hlines(np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
        ax.annotate('occluder', xy=(x_pos, -5),xytext=(x_pos, -8),ha='center', va='top', fontsize=9,arrowprops=dict(arrowstyle='->', color='k'),clip_on=False)
        ax.set_ylim([-12,12])
        ax.set_ylabel('x-gaze position (dva)')
        ax.set_xlabel('time since trial start (s)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlim((np.float64(-0.65), np.float64(3)))
ax.text(0.01, .98, f'sub=S{str(select_p+1).zfill(2)}', transform=ax.transAxes, fontsize=11, verticalalignment='top', horizontalalignment='left')

# add correct trajectory 
curr_traj_vis = list(pre_occ_traj)+list(np.ones(int(standard_occ_length+n_occ_frames[4]))*np.nan)+list(post_occ_traj)
ax.plot(traj_tv,curr_traj_vis,linestyle='-',color=colors[4])
ax.plot(np.array(traj_tv)[np.isnan(curr_traj_vis)],np.array(traj)[np.isnan(curr_traj_vis)],linestyle=':',color=colors[4])
#************************************
### (D) EYETRACKING -- group
#************************************

ax = shrink_axes(axs[1][1],0.95,0.8)

group_dat = np.array(toplot_group)
[ax.plot(epos.times,np.mean(group_dat[:,js_i,:],axis=0),color=colors[js_i]) for js_i in range(9)]
[ax.plot(occlusion_end_times[js_i],-12,'^',color=colors[js_i],markersize=10,zorder=10,clip_on=1) for js_i in range(9)]
ax.vlines(x_pos,-np.max(epos.metadata.occluder_start_pos_deg),np.max(epos.metadata.occluder_start_pos_deg),'k',linestyles='-',lw=2)
ax.hlines(-np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.hlines(np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.annotate('occluder', xy=(x_pos, -5),xytext=(x_pos, -8),ha='center', va='top', fontsize=9,arrowprops=dict(arrowstyle='->', color='k'),clip_on=False)
ax.set_ylim([-12,12])
ax.set_ylabel('x-gaze position (dva)')
ax.set_xlabel('time since trial start (s)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlim((np.float64(-0.65), np.float64(3)))
ax.text(0.01, .98, 'Average (n=20)', transform=ax.transAxes, fontsize=11, verticalalignment='top', horizontalalignment='left')

# add correct trajectory 
curr_traj_vis = list(pre_occ_traj)+list(np.ones(int(standard_occ_length+n_occ_frames[4]))*np.nan)+list(post_occ_traj)
ax.plot(traj_tv,curr_traj_vis,linestyle='-',color=colors[4])
ax.plot(np.array(traj_tv)[np.isnan(curr_traj_vis)],np.array(traj)[np.isnan(curr_traj_vis)],linestyle=':',color=colors[4])

### SAVE
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp2.png',dpi=300)
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp2.pdf')




Optimization terminated successfully.
         Current function value: 0.604165
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.344998
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.419134
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.256990
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.178730
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.343712
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.203169
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.329308
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.205236
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.345502
  

/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_91270/1816892816.py:136: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax_inset.boxplot([np.array(all_pse).flatten()], vert=False, widths=0.5,patch_artist=True,boxprops=dict(facecolor='white', color='k'),medianprops=dict(color='k'),whiskerprops=dict(color='k'),capprops=dict(color='k'),flierprops=dict(marker='x', color='k', markersize=4))


mean pse  = 0.19976162834067368, BF = 10^[-0.47223818]
[np.float64(0.3371023780586436)]


/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_91270/1816892816.py:183: RuntimeWarning: Mean of empty slice
  toplot_group_alljumps.append(np.nanmean(toplot_js,axis=0))
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_91270/1816892816.py:183: RuntimeWarning: Mean of empty slice
  toplot_group_alljumps.append(np.nanmean(toplot_js,axis=0))
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_91270/1816892816.py:183: RuntimeWarning: Mean of empty slice
  toplot_group_alljumps.append(np.nanmean(toplot_js,axis=0))


In [4]:
# bf for each condition compared to "true" trajectory during occlusion

bf_conds=[]

# eye-tracking was epoched from 500ms prior to trial onset to 2500 ms

# pre-occlusion = 49 frames at 60 Hz; 816.83 ms
# occlusion period - relative to pre-occlusion speed = 53 frames at 60 Hz; 883.51 ms
# post-occlusion = 49 frames at 60 Hz; 816.83 ms

# determine onset and offset of occlusion
baseline = 0.5
pre_post_occ_T = len(pre_occ_traj)*(1/epos.metadata['monitor_refreshrate'].unique()[0])
occ_T = (len(np.array(traj)[np.isnan(curr_traj_vis)]))*(1/epos.metadata['monitor_refreshrate'].unique()[0])
#occ_T = (len(np.array(traj_tv)[np.isnan(curr_traj_vis)]))*(1/epos.metadata['monitor_refreshrate'].unique()[0])

#index occlusion period out
gp_occ=group_dat[:,:,round((pre_post_occ_T+baseline)*1000):round((occ_T+pre_post_occ_T+baseline)*1000)]

#baseline to the "true" trajectory - currently plotted by frame; need to upsample to the ms

#1) upsample to match gp_occ samples
y_old=np.array(traj)[np.isnan(curr_traj_vis)]

# Original time points
t_old = np.linspace(0, 882, len(y_old))

# New time points (1 ms)
t_new = np.arange(gp_occ.shape[2])

# Shape-preserving interpolation
interp = interp1d(t_old, y_old, kind='linear')
# interp = PchipInterpolator(t_old, x_old)
y_new = interp(t_new)

#2) minus gaze trace from the true trajectory during the occlusion period for each conditon

gp_occ_base=gp_occ-y_new

#3) run bayes factors

for cond in range(9):
    bf_cond = calculate_bf(gp_occ_base[:,cond,:])

    bf_conds.append(bf_cond)




In [6]:
#plot the bayes factors for eye gaze deviation during occlusion for each of the 9 conditions 

plt.close('all')
fig,axs = plt.subplots(9,1,num=2,figsize=(a4_width, a4_height*0.7))

exponential_minmax=12
x_occ_lab=np.arange(1, len(y_new)+1)

# Labels for the subplots
labels = ['(too late max)', '', '', '(too late min)', '(on time)','(too early min)', '', '','(too early max)']
for i, (ax, label) in enumerate(zip(axs.flat, labels)):

    #plot bfs
    plot_bfs(fig,ax,x=x_occ_lab,y=bf_conds[i],n_cols = 500,bf_cmap="coolwarm",exponential_minmax=exponential_minmax)

    pos = ax.get_position()  
    x = pos.x0 - 0.05
    y = pos.y1
    fig.text(x, y, label, fontsize=10, fontweight='bold', va='top', ha='right')

    [ax.spines[i].set_visible(False) for i in ['top','right']]
    ax.tick_params(axis='y', which='major', labelsize=8)

    # Only show x-axis labels on the final subplot
    if i == len(axs) - 1:
        ax.set_xticks(np.arange(0, len(y_new)+1, 49))
        ax.set_xlabel('time since occlusion start (ms)')

    else:
        ax.set_xticks([])

    
    # Only add y-axis label on middle graph
    if i == 4:
        ax.set_ylabel('BF',fontsize=10)

############ GE NOTES - IF WE WANTED TO USE THIS FIGURE FOR SUPPLEMENTAL ############

# 1) color bar needs to be made skinnier
# 2) flip the subplots so too early comes first
# 3) cut off the x-axis at the reappearance time for each of the conditions - all information after that point is uninformative

# find the first timepoint when there is evidence for a difference between the true trajectory and the gaze position
bf_conds_arr = np.array(bf_conds)
ms_evid = np.argmax(bf_conds_arr > 6, axis=1)
ave_ms_idx=np.mean(ms_evid)

# find the timepont when the 6 conditions start to lag behind (average across subj and find first minus number)
gp_occ_ave = np.mean(gp_occ_base,axis=0)
idx_lag = np.argmax(gp_occ_ave < 0, axis=1)

#take the average lag of the first 6 conditions where the gaze lags behind the occluded object
ave_idx_lag=np.mean(idx_lag[:6])


### Relationship between gaze behaviour in Exp 1a and 2

In [7]:
# overlay eyetracking data of the "correct" trial with that of Exp 1a just to see whether they are dping the same thing
plt.close('all')
fig,ax = plt.subplots()

#### EXP 2
epos = all_epochs[0].copy()

group_dat = np.array(toplot_group)
ax.plot(epos.times,np.mean(group_dat[:,4,:],axis=0),color='r')
ax.plot(occlusion_end_times[4],-12,'^',color='r',markersize=10,zorder=10,clip_on=1)
ax.vlines(x_pos,-np.max(epos.metadata.occluder_start_pos_deg),np.max(epos.metadata.occluder_start_pos_deg),'k',linestyles='-',lw=2)
ax.hlines(-np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.hlines(np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.annotate('occluder', xy=(x_pos, -5),xytext=(x_pos, -8),ha='center', va='top', fontsize=9,arrowprops=dict(arrowstyle='->', color='k'),clip_on=False)
ax.set_ylim([-12,12])
ax.set_ylabel('x-gaze position (dva)')
ax.set_xlabel('time since trial start (s)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlim((np.float64(-0.65), np.float64(3)))
ax.text(0.01, .98, 'Average Exp 2 (n=20)', transform=ax.transAxes, fontsize=11, verticalalignment='top', horizontalalignment='left',color='r')

# add correct trajectory 
curr_traj_vis = list(pre_occ_traj)+list(np.ones(int(standard_occ_length+n_occ_frames[4]))*np.nan)+list(post_occ_traj)
ax.plot(traj_tv,curr_traj_vis,linestyle='-',color=colors[4])
ax.plot(np.array(traj_tv)[np.isnan(curr_traj_vis)],np.array(traj)[np.isnan(curr_traj_vis)],linestyle=':',color=colors[4])


###### EXP1a
colors_1a = sns.color_palette('mako',7)
bids_dir = '/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data'
subjids = [i for i in np.arange(1,21) if i!=9]
all_epochs_exp1a = read_data(bids_dir,subjids,task='exp1a')

sample_p = 0
epos_exp1a = all_epochs_exp1a[sample_p].copy()
x_idx = np.where([i=='xpos' for i in epos_exp1a.ch_names])[0][0]
epos_exp1a._data[epos_exp1a.metadata.direction==2,x_idx,:] = epos_exp1a._data[epos_exp1a.metadata.direction==2,x_idx,:]*-1

unique_stop_movement_time  = np.sort(epos_exp1a.metadata.stop_movement_time.unique())

for i,v in enumerate(unique_stop_movement_time):
    # plot mean x-gaze
    toplot_all = []
    for p in range(len(subjids)):
        epos_exp1a = all_epochs_exp1a[p].copy()
        x_idx = np.where([i=='xpos' for i in epos_exp1a.ch_names])[0][0]
        epos_exp1a._data[epos_exp1a.metadata.direction==2,x_idx,:] = epos_exp1a._data[epos_exp1a.metadata.direction==2,x_idx,:]*-1
        
        sel_epo = epos_exp1a[(epos_exp1a.metadata.stop_movement_time==v)]
        toplot_all.append(np.nanmean(sel_epo._data[:,x_idx],axis=0))
    toplot = np.mean(toplot_all,axis=0)
    stopping_idx_traj = np.argmin(np.abs(traj_tv-v))
    stopping_idx_epo = (epos_exp1a.times<=0).sum()+np.argmin(np.abs(epos_exp1a.times[(epos_exp1a.times>0)]-traj_tv[stopping_idx_traj]))
    ax.plot(sel_epo.times[:stopping_idx_epo],toplot[:stopping_idx_epo],color = colors_1a[i],lw=1,alpha=0.8)


gaze_error = []
for i,v in enumerate(unique_stop_movement_time):    
    # add marker for stopping position and connector lines
    stopping_idx_traj = np.argmin(np.abs(traj_tv-v))
    ax.plot(traj_tv[stopping_idx_traj],traj[stopping_idx_traj],marker='*',color=colors_1a[i],markeredgecolor=None,markersize=10,alpha=0.6)
    stopping_idx_epo = (epos.times<=0).sum()+np.argmin(np.abs(epos_exp1a.times[(epos_exp1a.times>0)]-traj_tv[stopping_idx_traj]))
    ax.vlines(epos_exp1a.times[stopping_idx_epo],toplot[stopping_idx_epo],traj[stopping_idx_traj],color=colors_1a[i],linestyles=':')
    
    gaze_error.append((np.array(toplot_all)[:,stopping_idx_epo])-traj[stopping_idx_traj])

ax.vlines(epos.metadata['occlusion_start_time'].unique()[0],-4.5,4.5,color='k')


plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp2-vs-exp1a.png',dpi=300)
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp2-vs-exp1a.pdf')


Reading /Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp1a/derivatives/preprocessed/sub-S01_exp1a_eyetracking-preproc-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -500.00 ...    1650.00 ms
        0 CTF compensation matrices available
Adding metadata with 21 columns
168 matching events found
No baseline correction applied
0 projection items activated
Reading /Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp1a/derivatives/preprocessed/sub-S02_exp1a_eyetracking-preproc-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -500.00 ...    1650.00 ms
        0 CTF compensation matrices available
Adding metadata with 21 columns
168 matching events found
No baseline correction applied
0 projection items activated
Reading /Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp1a/derivatives/preprocessed